# Data Preprocessing
This notebook involves cleaning, transforming, and preparing the data for modeling, including handling missing values, encoding categorical variables, and resampling dataset.

## Import Dependencies

In [11]:
import os
import warnings

import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.utils import resample

from mappers import to_numeric, encode, get_keys

warnings.filterwarnings("ignore")

## Data Loading

In [2]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
path = os.path.join(root, "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

## Data Cleaning

#### Remove unnecessary features
1. `Category URL`

2. `Service URL`

3. `Offer URL`

4. `Offer Name`

5. `Owner URL`

6. `Owner Name`


In [3]:
unnecessary_features = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]

dataset.drop(columns=unnecessary_features, inplace=True)

#### Convert text-based values to numeric values
1. Time: `Duration`, `Offer Response Time`, and `Owner Response Time`.

2. Percentage: `Owner Completion Rate`.

3. Boolean: `Owner Verified`.

4. Money: `Price`.


In [4]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")

dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)

dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)

dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time", "Owner Level"])

## Encoding Categorical Features

In [5]:
category_encoded = pd.DataFrame(dataset["Category Name"].apply(lambda value: encode(value, "Category Name")).tolist(), columns=[key for key in get_keys("Category Name")])

service_encoded = pd.DataFrame(dataset["Service Name"].apply(lambda value: encode(value, "Service Name")).tolist(), columns=[key for key in get_keys("Service Name")])

dataset = dataset.drop(columns=["Category Name", "Service Name"])

dataset = pd.concat([dataset, category_encoded, service_encoded], axis=1)

## Handling Missing Values

### Imputing missing values using KNNImputer

*After many trails the best value of `n_neighbors` is 3.*

In [6]:
imputer = KNNImputer(n_neighbors=3)

columns_to_impute = ["Offer Response Time", "Owner Response Time", "Owner Completion Rate"]

dataset[columns_to_impute] = imputer.fit_transform(dataset[columns_to_impute])

In [13]:
dataset.shape

(7818, 303)

## Save the Cleaned Dataset

In [10]:
path = os.path.join(root, "clean.csv")

dataset.to_csv(path_or_buf=path, index=False)

## Balancing Dataset by Resampling (Upsample) for Price Ranges

In [14]:
max_count = dataset["Price"].value_counts()[5]

balanced_dataset = pd.concat([resample(dataset[dataset["Price"] == value], replace=True, n_samples=max_count, random_state=42) for value in range(5, 51, 5)])

In [15]:
balanced_dataset.shape

(43410, 303)

## Save the Balanced Dataset

In [18]:
path = os.path.join(root, "balanced.csv")

balanced_dataset.to_csv(path_or_buf=path, index=False)